In [ ]:
import torch
from training_loop import training_loop
from extract_embedding import get_train_test_loaders
from qwen_tts.core.models import BasicSpeakerEncoder
import optuna

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



********
********
 


2026-04-24 13:25:26.305354715 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


In [2]:
# Setup the data loaders
train_loader, test_loader = get_train_test_loaders()


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB
Loading Qwen3-TTS model...


Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 34663.67it/s]


Qwen3 speaker encoder loaded on cuda
train-clean-100 already extracted
dev-clean already extracted
Loaded 12526 samples
Loaded 1953 samples
Train: 12526 samples, 391 batches
Test:  1953 samples, 62 batches


In [5]:

def objective(trial, train_loader, test_loader):
    # 1. Define the hyperparameter search space
    # Optuna will sample values from these ranges for each trial
    num_epochs = trial.suggest_int("num_epochs", 10, 50)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    alpha = trial.suggest_float("loss_alpha", 0.1, 0.9) 
    
    # Assemble the dictionary for the training loop
    hyperparams = {
        "num_epochs": num_epochs,
        "save_every": 20,       # Prevent excessive checkpoint saving during sweeps
        "lr": lr,
        "loss_alpha": alpha,
        "weight_decay": weight_decay,
        "model_name": f"BasicSpeakerEncoder_trial_{trial.number}"
    }

    # 2. Setup Device and model
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    # Setup the model
    model = BasicSpeakerEncoder().to(device)
    
        
    # 3. Execute the training loop
    # We catch the returned best test cosine similarity
    best_test_cosine = training_loop(model, train_loader, test_loader, device, hyperparams)
    
    # Optuna will try to MAXIMIZE this returned value
    return best_test_cosine


In [6]:
sampler = optuna.samplers.TPESampler(n_startup_trials=5)

study = optuna.create_study(direction="maximize", study_name="Speaker_Encoder_Distillation", sampler=sampler)
study.optimize(lambda trial: objective(trial, train_loader, test_loader), n_trials=20)
    
print("\n=================================")
print("Best Trial Found:")
print(f"Best Test Cosine Similarity: {study.best_trial.value:.4f}")
print("Optimal Hyperparameters: ")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")

[I 2026-04-24 13:31:53,806] A new study created in memory with name: Speaker_Encoder_Distillation


Epoch   1/27 | Train loss=0.1513 cos=0.7220 | Test loss=0.1187 cos=0.9596 *best* | 6.9s
Epoch   2/27 | Train loss=0.1173 cos=0.9693 | Test loss=0.1174 cos=0.9692 *best* | 6.9s
Epoch   3/27 | Train loss=0.1165 cos=0.9747 | Test loss=0.1169 cos=0.9730 *best* | 7.0s
Epoch   4/27 | Train loss=0.1162 cos=0.9773 | Test loss=0.1166 cos=0.9751 *best* | 7.0s
Epoch   5/27 | Train loss=0.1160 cos=0.9788 | Test loss=0.1164 cos=0.9765 *best* | 7.1s
Epoch   6/27 | Train loss=0.1158 cos=0.9800 | Test loss=0.1163 cos=0.9776 *best* | 7.3s
Epoch   7/27 | Train loss=0.1157 cos=0.9809 | Test loss=0.1162 cos=0.9785 *best* | 7.3s
Epoch   8/27 | Train loss=0.1156 cos=0.9816 | Test loss=0.1161 cos=0.9790 *best* | 7.5s
Epoch   9/27 | Train loss=0.1155 cos=0.9821 | Test loss=0.1160 cos=0.9797 *best* | 7.5s
Epoch  10/27 | Train loss=0.1154 cos=0.9827 | Test loss=0.1159 cos=0.9802 *best* | 7.6s
Epoch  11/27 | Train loss=0.1154 cos=0.9831 | Test loss=0.1159 cos=0.9805 *best* | 7.7s
Epoch  12/27 | Train loss=0.1153

[I 2026-04-24 13:35:17,161] Trial 0 finished with value: 0.9835293341067529 and parameters: {'num_epochs': 27, 'lr': 2.3511577730146884e-05, 'weight_decay': 0.00030300038847916235, 'loss_alpha': 0.12262515595555962}. Best is trial 0 with value: 0.9835293341067529.


Epoch  27/27 | Train loss=0.1150 cos=0.9861 | Test loss=0.1155 cos=0.9835 *best* | 7.7s

Done! Best test cosine similarity: 0.9835
Epoch   1/28 | Train loss=0.1402 cos=0.8470 | Test loss=0.0890 cos=0.9707 *best* | 7.7s
Epoch   2/28 | Train loss=0.0865 cos=0.9764 | Test loss=0.0868 cos=0.9760 *best* | 7.7s
Epoch   3/28 | Train loss=0.0851 cos=0.9799 | Test loss=0.0859 cos=0.9781 *best* | 7.8s
Epoch   4/28 | Train loss=0.0843 cos=0.9817 | Test loss=0.0852 cos=0.9798 *best* | 7.8s
Epoch   5/28 | Train loss=0.0838 cos=0.9831 | Test loss=0.0848 cos=0.9808 *best* | 7.9s
Epoch   6/28 | Train loss=0.0834 cos=0.9840 | Test loss=0.0844 cos=0.9818 *best* | 7.9s
Epoch   7/28 | Train loss=0.0831 cos=0.9847 | Test loss=0.0842 cos=0.9823 *best* | 7.8s
Epoch   8/28 | Train loss=0.0829 cos=0.9853 | Test loss=0.0840 cos=0.9828 *best* | 7.8s
Epoch   9/28 | Train loss=0.0827 cos=0.9857 | Test loss=0.0838 cos=0.9832 *best* | 7.8s
Epoch  10/28 | Train loss=0.0825 cos=0.9861 | Test loss=0.0837 cos=0.9835 *be

[I 2026-04-24 13:38:58,279] Trial 1 finished with value: 0.985357488355329 and parameters: {'num_epochs': 28, 'lr': 6.100806806598487e-05, 'weight_decay': 0.0003501676729757289, 'loss_alpha': 0.40431687698123386}. Best is trial 1 with value: 0.985357488355329.


...

Epoch  46/46 | Train loss=0.0364 cos=0.9863 | Test loss=0.0385 cos=0.9837 *best* | 8.0s

Done! Best test cosine similarity: 0.9837
Epoch   1/37 | Train loss=0.0963 cos=0.9425 | Test loss=0.0814 cos=0.9745 *best* | 7.9s
Epoch   2/37 | Train loss=0.0788 cos=0.9798 | Test loss=0.0792 cos=0.9790 *best* | 7.9s
Epoch   3/37 | Train loss=0.0773 cos=0.9828 | Test loss=0.0781 cos=0.9814 *best* | 7.9s
Epoch   4/37 | Train loss=0.0766 cos=0.9845 | Test loss=0.0777 cos=0.9823 *best* | 7.8s
Epoch   5/37 | Train loss=0.0760 cos=0.9856 | Test loss=0.0772 cos=0.9833 *best* | 8.0s
Epoch   6/37 | Train loss=0.0756 cos=0.9865 | Test loss=0.0769 cos=0.9840 *best* | 7.9s
Epoch   7/37 | Train loss=0.0753 cos=0.9872 | Test loss=0.0767 cos=0.9845 *best* | 7.9s
Epoch   8/37 | Train loss=0.0750 cos=0.9877 | Test loss=0.0767 cos=0.9844 | 7.7s
Epoch   9/37 | Train loss=0.0748 cos=0.9882 | Test loss=0.0763 cos=0.9852 *best* | 7.9s
Epoch  10/37 | Train loss=0.0746 cos=0.9885 | Test loss=0.0762 cos=0.9854 *best* | 7

[I 2026-04-24 15:01:55,984] Trial 18 finished with value: 0.9871730439124569 and parameters: {'num_epochs': 37, 'lr': 0.00033146433091739137, 'weight_decay': 3.8206465785387185e-06, 'loss_alpha': 0.46259612917288945}. Best is trial 13 with value: 0.9873257413987191.


Epoch  37/37 | Train loss=0.0731 cos=0.9919 | Test loss=0.0754 cos=0.9871 | 7.9s

Done! Best test cosine similarity: 0.9872
Epoch   1/46 | Train loss=0.0757 cos=0.9583 | Test loss=0.0653 cos=0.9750 *best* | 7.8s
Epoch   2/46 | Train loss=0.0620 cos=0.9802 | Test loss=0.0619 cos=0.9806 *best* | 8.5s
Epoch   3/46 | Train loss=0.0597 cos=0.9839 | Test loss=0.0605 cos=0.9828 *best* | 7.9s
Epoch   4/46 | Train loss=0.0587 cos=0.9857 | Test loss=0.0601 cos=0.9834 *best* | 8.4s
Epoch   5/46 | Train loss=0.0581 cos=0.9866 | Test loss=0.0596 cos=0.9842 *best* | 7.7s
Epoch   6/46 | Train loss=0.0576 cos=0.9874 | Test loss=0.0595 cos=0.9844 *best* | 7.8s
Epoch   7/46 | Train loss=0.0572 cos=0.9880 | Test loss=0.0594 cos=0.9845 *best* | 7.9s
Epoch   8/46 | Train loss=0.0570 cos=0.9883 | Test loss=0.0591 cos=0.9850 *best* | 7.8s
Epoch   9/46 | Train loss=0.0568 cos=0.9887 | Test loss=0.0591 cos=0.9850 *best* | 7.6s
Epoch  10/46 | Train loss=0.0566 cos=0.9890 | Test loss=0.0591 cos=0.9850 | 7.7s
Epo

[I 2026-04-24 15:07:55,415] Trial 19 finished with value: 0.9871674881827447 and parameters: {'num_epochs': 46, 'lr': 0.004088955007975085, 'weight_decay': 3.863952956411941e-05, 'loss_alpha': 0.6137686025714704}. Best is trial 13 with value: 0.9873257413987191.


Epoch  46/46 | Train loss=0.0541 cos=0.9929 | Test loss=0.0578 cos=0.9872 | 7.9s

Done! Best test cosine similarity: 0.9872

Best Trial Found:
Best Test Cosine Similarity: 0.9873
Optimal Hyperparameters: 
num_epochs: 50
lr: 0.0003936063881351834
weight_decay: 3.0611623852523348e-06
loss_alpha: 0.5700387065413405
